# Notebook 034: ACID CRUD Operations on Delta: Insert, Update and Delete Mechanics

**Module:** Module 04: Delta Lake Architecture, ACID Operations & Time Travel  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 14: Master Unity Catalog & Video 20: Finance Domain E2E Project  

---

## 1. Business Scenario & Real-World Context
A GDPR/CCPA privacy compliance pipeline requires modifying existing customer records: deleting individuals exercising 'Right to be Forgotten' and updating consent preferences without taking the database offline.

## 2. Technical Requirement & Problem Statement
Execute ACID CRUD operations on Delta Lake using `DeltaTable` API and Spark SQL: append new records (`insert`), selectively modify attributes (`update`), and purge records (`delete`). Inspect file modifications and tombstone markers.

## 3. Spark & Databricks Architecture Notes
Delta Lake files are immutable. An UPDATE or DELETE operation reads existing Parquet files, rewrites modified rows into brand-new Parquet files, and writes tombstone `remove` actions into the Delta transaction log. The old files remain on disk for Time Travel until removed by `VACUUM`.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_034_ACID_CRUD_Operations_on_Delta_Insert,_Update_and_Delete_Mechanics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Initializing Customer Privacy Table in Delta
Create customers table with consent flags.

In [ ]:
from delta.tables import DeltaTable

delta_privacy_path = "/tmp/delta_privacy_034"
if Path(delta_privacy_path).exists():
    shutil.rmtree(delta_privacy_path)

privacy_data = [
    ("U-01", "alice@corp.com", True, "2023-01-10"),
    ("U-02", "bob@spam.com", False, "2023-02-15"),
    ("U-03", "charlie@privacy.org", True, "2023-03-20")
]

df_priv = spark.createDataFrame(privacy_data, ["user_id", "email", "marketing_consent", "created_date"])
df_priv.write.format("delta").mode("overwrite").save(delta_privacy_path)

delta_table = DeltaTable.forPath(spark, delta_privacy_path)
delta_table.toDF().show()

## 6. Executing ACID UPDATE and DELETE via DeltaTable API
Update consent flags and execute GDPR deletion.

In [ ]:
# 1. Update: Revoke consent for bob@spam.com
delta_table.update(
    condition=F.col("email") == "bob@spam.com",
    set={"marketing_consent": F.lit(False)}
)

# 2. Delete: Purge user exercising GDPR Right to be Forgotten (U-03)
delta_table.delete(condition=F.col("user_id") == "U-03")

print("Table State after UPDATE and DELETE:")
delta_table.toDF().show()

## 7. Inspecting Commit History & File Rewrites
Observe UPDATE and DELETE operations logged as distinct atomic transactions.

In [ ]:
df_hist = delta_table.history()
df_hist.select("version", "operation", "operationMetrics").show(truncate=False)

print(f"Current version of table: {df_hist.first()['version']}")

## 9. Key Architectural Takeaways
- **Delta Lake brings row-level UPDATE and DELETE capabilities to Apache Spark without rewriting whole partitions manually.**
- **Updates and Deletes create new Parquet files and log `remove` actions in `_delta_log/` without immediately deleting physical files.**
- **All operations are fully ACID-compliant: concurrent readers always observe a consistent snapshot.**

## 10. Senior Data Engineering Interview Questions
1. Why are UPDATE and DELETE operations in Delta Lake relatively expensive compared to simple append writes?
2. How does Copy-on-Write (CoW) in Delta Lake compare to Merge-on-Read (MoR) in Apache Iceberg/Hudi?
3. What happens to the physical Parquet files on disk when a row is deleted in Delta Lake?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
